# Fare vs Reliability Relationship Analysis — Bristol (First Bus)
### Full Pipeline: Parsing -> Cleaning -> PySpark

This notebook takes you from raw BODS XML/CSV files all the way to cleaned, partitioned Spark DataFrames saved as Parquet, ready for joining and modelling.

**Folder structure expected (same folder as this notebook):**
```
timetables/    <- unzipped Timetables XML files
fares/         <- unzipped Fares XML files
catalogue/     <- contains disruptions_data_catalogue.csv
```


## Part A — Parsing & Flattening (Pandas)

### 1. Imports

In [ ]:
import xml.etree.ElementTree as ET
import glob, re, os
import pandas as pd

### 2. Parse Timetables (TransXChange XML)

Flattens each VehicleJourney into one row per stop, with cumulative scheduled time computed from `DepartureTime` + `RunTime` on each `JourneyPatternTimingLink`.

In [ ]:
ns = {'txc': 'http://www.transxchange.org.uk/'}

def parse_runtime(rt):
    if not rt:
        return 0
    h = re.search(r'(\d+)H', rt); m = re.search(r'(\d+)M', rt); s = re.search(r'(\d+)S', rt)
    total = 0
    if h: total += int(h.group(1)) * 3600
    if m: total += int(m.group(1)) * 60
    if s: total += int(s.group(1))
    return total

def parse_timetable_file(filepath):
    tree = ET.parse(filepath)
    root = tree.getroot()

    stop_names = {}
    for sp in root.findall('.//txc:StopPoints/txc:AnnotatedStopPointRef', ns):
        ref = sp.find('txc:StopPointRef', ns)
        name = sp.find('txc:CommonName', ns)
        if ref is not None:
            stop_names[ref.text] = name.text if name is not None else None

    operators = {}
    for op in root.findall('.//txc:Operators/txc:Operator', ns):
        noc = op.find('txc:NationalOperatorCode', ns)
        operators[op.get('id')] = noc.text if noc is not None else None

    services = root.findall('.//txc:Services/txc:Service', ns)
    service_code, line_name = None, None
    if services:
        sc = services[0].find('txc:ServiceCode', ns)
        service_code = sc.text if sc is not None else None
        ln = services[0].find('.//txc:LineName', ns)
        line_name = ln.text if ln is not None else None

    journey_patterns = {}
    for jp in root.findall('.//txc:JourneyPattern', ns):
        opref = jp.find('txc:OperatorRef', ns)
        routeref = jp.find('txc:RouteRef', ns)
        direction = jp.find('txc:Direction', ns)
        jps_refs = jp.find('txc:JourneyPatternSectionRefs', ns)
        journey_patterns[jp.get('id')] = {
            'operator_ref': opref.text if opref is not None else None,
            'route_ref': routeref.text if routeref is not None else None,
            'direction': direction.text if direction is not None else None,
            'jps_ref': jps_refs.text if jps_refs is not None else None
        }

    jps_dict = {}
    for jps in root.findall('.//txc:JourneyPatternSections/txc:JourneyPatternSection', ns):
        links = []
        for link in jps.findall('txc:JourneyPatternTimingLink', ns):
            frm = link.find('txc:From', ns); to = link.find('txc:To', ns); rt = link.find('txc:RunTime', ns)
            from_stop = frm.find('txc:StopPointRef', ns).text if frm is not None else None
            to_stop = to.find('txc:StopPointRef', ns).text if to is not None else None
            links.append((from_stop, to_stop, parse_runtime(rt.text if rt is not None else None)))
        jps_dict[jps.get('id')] = links

    rows = []
    for vj in root.findall('.//txc:VehicleJourneys/txc:VehicleJourney', ns):
        vj_code = vj.find('txc:PrivateCode', ns)
        dep_time = vj.find('txc:DepartureTime', ns)
        jp_ref = vj.find('txc:JourneyPatternRef', ns)
        op_ref = vj.find('txc:OperatorRef', ns)
        days = [d.tag.split('}')[-1] for d in vj.findall('.//txc:DaysOfWeek/*', ns)]

        jp_info = journey_patterns.get(jp_ref.text if jp_ref is not None else None, {})
        links = jps_dict.get(jp_info.get('jps_ref'), [])

        if dep_time is not None and dep_time.text:
            h, m, s = map(int, dep_time.text.split(':'))
            cum = h * 3600 + m * 60 + s
        else:
            cum = None

        noc = operators.get(op_ref.text if op_ref is not None else jp_info.get('operator_ref'))

        if links:
            first_stop = links[0][0]
            rows.append(dict(service_code=service_code, line_name=line_name, operator_noc=noc,
                              vehicle_journey_code=vj_code.text if vj_code is not None else None,
                              direction=jp_info.get('direction'), days_of_week=','.join(days),
                              stop_sequence=0, stop_point_ref=first_stop,
                              stop_name=stop_names.get(first_stop), scheduled_time_sec=cum,
                              source_file=os.path.basename(filepath)))
            seq = 0
            for (frm, to, rt) in links:
                cum = cum + rt if cum is not None else None
                seq += 1
                rows.append(dict(service_code=service_code, line_name=line_name, operator_noc=noc,
                                  vehicle_journey_code=vj_code.text if vj_code is not None else None,
                                  direction=jp_info.get('direction'), days_of_week=','.join(days),
                                  stop_sequence=seq, stop_point_ref=to,
                                  stop_name=stop_names.get(to), scheduled_time_sec=cum,
                                  source_file=os.path.basename(filepath)))
    return rows

def build_timetables_df(folder):
    all_rows = []
    for f in glob.glob(os.path.join(folder, '*.xml')):
        try:
            all_rows.extend(parse_timetable_file(f))
        except Exception as e:
            print(f'Failed on {f}: {e}')
    df = pd.DataFrame(all_rows)
    df['scheduled_time'] = pd.to_timedelta(df['scheduled_time_sec'], unit='s').astype(str).str.split(' ').str[-1]
    return df

In [ ]:
timetables_df = build_timetables_df('timetables')
print('Total rows:', len(timetables_df))
timetables_df.head(10)

In [ ]:
timetables_df.to_csv('timetables_flat.csv', index=False)

### 3. Parse Fares (NeTEx XML)

Each fare file is a single ticket product. Route, direction, and ticket type are embedded in the filename; price is read from the `<Amount>` tag.

In [ ]:
nsf = {'netex': 'http://www.netex.org.uk/netex'}

def parse_fare_file(filepath):
    filename = os.path.basename(filepath)
    tree = ET.parse(filepath)
    root = tree.getroot()

    amounts = [a.text for a in root.findall('.//netex:Amount', nsf)]

    stem = filename.replace('.xml', '')
    parts = stem.split('_')
    operator = parts[0] if len(parts) > 0 else None
    route = parts[1] if len(parts) > 1 else None
    direction = parts[2] if len(parts) > 2 else None
    ticket_type = parts[3] if len(parts) > 3 else None

    rows = []
    if amounts:
        for amt in amounts:
            rows.append(dict(operator=operator, route=route, direction=direction,
                              ticket_type=ticket_type, price_gbp=float(amt),
                              source_file=filename))
    else:
        rows.append(dict(operator=operator, route=route, direction=direction,
                          ticket_type=ticket_type, price_gbp=None, source_file=filename))
    return rows

def build_fares_df(folder):
    all_rows = []
    fail_count = 0
    for f in glob.glob(os.path.join(folder, '*.xml')):
        try:
            all_rows.extend(parse_fare_file(f))
        except Exception as e:
            fail_count += 1
    df = pd.DataFrame(all_rows)
    print(f'Failed files: {fail_count}')
    return df

In [ ]:
fares_df = build_fares_df('fares')
print('Total fare rows:', len(fares_df))
print('Null prices:', fares_df['price_gbp'].isna().sum())
print('Unique routes:', fares_df['route'].nunique())
fares_df.head(10)

In [ ]:
fares_df.to_csv('fares_flat.csv', index=False)

### 4. Load Disruptions (from BODS data catalogue export)

The `disruptions_data_catalogue.csv` already contains real disruption event records — no XML parsing needed.

In [ ]:
disruptions_df = pd.read_csv('catalogue/disruptions_data_catalogue.csv')
disruptions_df = disruptions_df[disruptions_df['Organisation'] == 'West of England']
print('West of England disruptions:', len(disruptions_df))
disruptions_df.head(10)

### 5. Sanity check — route overlap across all 3 datasets

In [ ]:
tt_routes = set(timetables_df['line_name'].dropna().astype(str).unique())
fr_routes = set(fares_df['route'].dropna().astype(str).unique())
di_routes = set(disruptions_df['Services affected'].dropna().astype(str).unique())

print('Timetable routes:', sorted(tt_routes))
print('Fares routes (sample):', sorted(fr_routes)[:20])
print('Disruption routes:', sorted(di_routes))

print('\nTimetables & Fares overlap:', len(tt_routes & fr_routes))
print('Timetables & Disruptions overlap:', len(tt_routes & di_routes))

## Part B — PySpark Loading, Cleaning & Partitioning

Now we move the three flattened tables into PySpark for the large-scale cleaning, partitioning, caching, and (later) joins/ML — satisfying the brief's PySpark requirement.

### 6. Start Spark session

In [ ]:
import os
os.environ["PYSPARK_PYTHON"] = "python"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

spark = SparkSession.builder \
    .appName("FareReliabilityAnalysis") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark

### 7. Load the three flattened files into Spark DataFrames

In [ ]:
timetables_schema = StructType([
    StructField("service_code", StringType(), True),
    StructField("line_name", StringType(), True),
    StructField("operator_noc", StringType(), True),
    StructField("vehicle_journey_code", StringType(), True),
    StructField("direction", StringType(), True),
    StructField("days_of_week", StringType(), True),
    StructField("stop_sequence", IntegerType(), True),
    StructField("stop_point_ref", StringType(), True),
    StructField("stop_name", StringType(), True),
    StructField("scheduled_time_sec", DoubleType(), True),
    StructField("source_file", StringType(), True),
    StructField("scheduled_time", StringType(), True),
])

timetables_spark = spark.read.csv("timetables_flat.csv", header=True, schema=timetables_schema)

fares_schema = StructType([
    StructField("operator", StringType(), True),
    StructField("route", StringType(), True),
    StructField("direction", StringType(), True),
    StructField("ticket_type", StringType(), True),
    StructField("price_gbp", DoubleType(), True),
    StructField("source_file", StringType(), True),
])

fares_spark = spark.read.csv("fares_flat.csv", header=True, schema=fares_schema)

disruptions_spark = spark.read.csv("catalogue/disruptions_data_catalogue.csv", header=True, inferSchema=True)
disruptions_spark = disruptions_spark.filter(F.col("Organisation") == "West of England")

print("Timetables rows:", timetables_spark.count())
print("Fares rows:", fares_spark.count())
print("Disruptions rows:", disruptions_spark.count())

### 8. Clean each DataFrame

In [ ]:
# --- Timetables cleaning ---
timetables_clean = (
    timetables_spark
    .filter(F.col("stop_point_ref").isNotNull())
    .filter(F.col("line_name").isNotNull())
    .withColumn("line_name", F.trim(F.col("line_name")))
    .dropDuplicates(["service_code", "vehicle_journey_code", "stop_sequence"])
)
print("Timetables - raw:", timetables_spark.count(), "| clean:", timetables_clean.count())

# --- Fares cleaning ---
fares_clean = (
    fares_spark
    .filter(F.col("price_gbp").isNotNull())
    .filter(F.col("price_gbp") > 0)
    .withColumn("route", F.trim(F.col("route")))
    .dropDuplicates(["operator", "route", "direction", "ticket_type"])
)
print("Fares - raw:", fares_spark.count(), "| clean:", fares_clean.count())

# --- Disruptions cleaning ---
disruptions_clean = (
    disruptions_spark
    .filter(F.col("Services affected").isNotNull())
    .withColumnRenamed("Services affected", "route")
    .withColumn("route", F.trim(F.col("route").cast("string")))
)
print("Disruptions (West of England) - raw:", disruptions_spark.count(), "| clean:", disruptions_clean.count())

### 9. Partitioning + Caching (Big Data requirement)

Repartition Timetables into at least 4 partitions and cache — required for the brief's Spark UI evidence.

In [ ]:
timetables_clean = timetables_clean.repartition(8, "line_name")
timetables_clean.cache()
fares_clean.cache()
disruptions_clean.cache()

print("Timetables partitions:", timetables_clean.rdd.getNumPartitions())

In [ ]:
timetables_clean.show(5, truncate=False)
fares_clean.show(5, truncate=False)
disruptions_clean.show(5, truncate=False)

### 10. Save cleaned DataFrames as Parquet (efficient format for later joins/ML)

In [ ]:
timetables_clean.write.mode("overwrite").parquet("output/timetables_clean.parquet")
fares_clean.write.mode("overwrite").parquet("output/fares_clean.parquet")
disruptions_clean.write.mode("overwrite").parquet("output/disruptions_clean.parquet")

print("Cleaning complete. Parquet files written to /output.")

## Next steps

- Join Timetables <-> Fares on `route`
- Join Timetables <-> Disruptions on `route`
- Engineer a reliability feature (e.g. disruption count per route, or trips-per-route vs disruptions-per-route ratio)
- Confirm total merged row count against the 100,000-record threshold (already well exceeded from Timetables alone: 500k+ rows)
- Move to feature engineering + ML modelling (regression: fare -> reliability score)
